# RecBinarySearch and ItBinarySearch (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/RecBinarySearch_ItBinarySearch_cpp.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `RecBinarySearch` and `ItBinarySearch`: the algorithms in C++, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/RecBinarySearch_ItBinarySearch_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/RecBinarySearch_ItBinarySearch_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `FibRec` (PascalCase).

## Tail recursion: RecBinarySearch and ItBinarySearch

A recursive call is a *tail call* if it is the last operation and its result is returned as it is: it can be replaced by an update of the parameters and a jump to the beginning, that is, by a loop (loop condition = negation of the base case; tail call = simultaneous assignment of the parameters). The result is the same, the number of steps has the same order of growth and the extra memory goes down from $\Theta(\text{depth})$ to $\Theta(1)$.

**RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: a sorted array, two indices $l$, $r$ and a value $m$. *Output*: an index of $m$ in $a[l..r]$ or $-1$. `ItBS(a, 1, n, m)` is the iterative binary search of the chapter "Search". In the pseudo-code of the chapter they are called `RecBinarySearch` and `ItBinarySearch`.

On the array $a=[3,9,11,12,12,16,19]$ of the chapter we search for $m=12,10,3,19,100$: for each value the returned index (from 0), the calls of `RecBS` and the iterations of `ItBS`.

In [1]:
%%writefile rec_it_binary_search.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int rbsCalls = 0;        // calls of RecBS
int bsIters = 0;         // iterations of ItBS

// RecBS(a, l, r, m)
int RecBS(int *a, int l, int r, int m)
{
    rbsCalls++;
    if (l > r)
    {
        return -1;
    }
    int mid = l + (r - l) / 2;
    if (a[mid] == m)
    {
        return mid;
    }
    if (a[mid] < m)
    {
        return RecBS(a, mid + 1, r, m);     // tail call
    }
    else
    {
        return RecBS(a, l, mid - 1, m);     // tail call
    }
}

// ItBS(a, l, r, m)
int ItBS(int *a, int l, int r, int m)
{
    while (l <= r)                          // negation of the base case l > r
    {
        bsIters++;
        int mid = l + (r - l) / 2;
        if (a[mid] == m)
        {
            return mid;                     // the second base case stays in the loop
        }
        if (a[mid] < m)
        {
            l = mid + 1;
        }
        else
        {
            r = mid - 1;
        }
    }
    return -1;
}

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

#ifndef NOMAIN
int main()
{
    int a[] = {3, 9, 11, 12, 12, 16, 19};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("%4s %6s %6s %6s %11s\n", "m", "RecBS", "calls", "ItBS", "iterations");
    int ms[] = {12, 10, 3, 19, 100};
    for (int m : ms)
    {
        rbsCalls = 0;
        int r1 = RecBS(a, 0, n - 1, m);
        bsIters = 0;
        int r2 = ItBS(a, 0, n - 1, m);
        printf("%4d %6d %6d %6d %11d   %s\n", m, r1, rbsCalls, r2, bsIters, r1 == r2 ? "OK" : "ERROR");
    }
    return 0;
}
#endif

Writing rec_it_binary_search.cpp


In [2]:
!g++ -std=c++17 -O2 -o rec_it_binary_search rec_it_binary_search.cpp && ./rec_it_binary_search

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Try it.** Search for other values, present or absent, in another sorted array: compare the calls of `RecBS` with the iterations of `ItBS`.

In [3]:
%%writefile prova_rec_it_binary_search.cpp
#define NOMAIN
#include "rec_it_binary_search.cpp"

int main()
{
    // Search for other values, present or absent: compare the calls of RecBS with the iterations of ItBS.
    int b[] = {2, 5, 8, 11, 14, 17, 20, 23};
    int n = 8;
    int ms[] = {14, 1, 30};
    for (int m : ms)
    {
        rbsCalls = 0;
        int r1 = RecBS(b, 0, n - 1, m);
        bsIters = 0;
        int r2 = ItBS(b, 0, n - 1, m);
        printf("%d %d %d %d %d\n", m, r1, rbsCalls, r2, bsIters);
    }
    return 0;
}

Writing prova_rec_it_binary_search.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_rec_it_binary_search prova_rec_it_binary_search.cpp && ./prova_rec_it_binary_search

14 4 3 4 3
1 -1 4 -1 3
30 -1 5 -1 4


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).